# Spark Voice Studio

Connect to the DGX Spark, download and load a voice model, record your voice, fine-tune, play back text, and tune on keywords. Each section runs one of the `stepN_*.py` scripts; edit the `ARGS` line and run the cell. The recorder (step 4) and the player (step 6) need a terminal-style prompt, which Jupyter provides through `input()`.

Edit `studio.yaml` first: Spark host, user, key, and your name as owner.

In [ ]:
import subprocess, sys, json
from pathlib import Path
ROOT = Path.cwd()

def run(script, args=''):
    """Run a step script with live output in the notebook."""
    cmd = [sys.executable, str(ROOT / script), *args.split()]
    print('$', ' '.join(cmd[1:]))
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, cwd=ROOT)
    for line in p.stdout: print(line, end='')
    return p.wait()

def status():
    p = ROOT / 'state.json'
    print(json.dumps(json.loads(p.read_text()), indent=1) if p.exists() else 'no state yet')

status()

## Step 1 - Connect to the DGX Spark.

Checks SSH, the GPU, and the Python environment on the Spark; creates the studio venv there with the
cu130 PyTorch wheels (sm_120 kernels run on the GB10's sm_121), the engine packages and the model server
dependencies; syncs this folder to the Spark; records the result in state.json.

    python step1_connect_spark.py            # connect, check, install what is missing
    python step1_connect_spark.py --check    # connect and report only

In [ ]:
ARGS = ''
run('step1_connect_spark.py', ARGS)

## Step 2 - Download the models to the Spark.

Pulls the Qwen3-TTS Base model (the voice-cloning and fine-tuning base), its 12 Hz tokenizer (needed to
prepare training data), and Whisper (for transcripts of your recordings) into <remote_home>/models.
Also clones the Qwen3-TTS repository, whose finetuning/ folder holds the training scripts.

    python step2_download_model.py                 # models from studio.yaml
    python step2_download_model.py --tts Qwen/Qwen3-TTS-12Hz-0.6B-Base

In [ ]:
ARGS = ''
run('step2_download_model.py', ARGS)

## Step 3 - Load the model into the Spark's memory.

Starts the model server on the Spark if it is not running, then tells it which model to hold:
  --mode clone   (default) the Base model with a voice prompt from data/voice/ref.wav (works before any training)
  --mode custom  a fine-tuned checkpoint from step 5 (checkpoints/<run>/checkpoint-epoch-N)

    python step3_load_model.py
    python step3_load_model.py --mode custom --path checkpoints/run1/checkpoint-epoch-3
    python step3_load_model.py --stop

In [ ]:
ARGS = '--mode clone'
run('step3_load_model.py', ARGS)

## Step 4 - Capture your voice for training.

Runs where the microphone is (your Mac). Shows one sentence at a time from prompts/capture_sentences.txt,
records it, trims silence, saves data/voice/uttNNNN.wav with its transcript uttNNNN.txt, and picks the
cleanest 10 to 15 s clip as data/voice/ref.wav (the reference every training sample and the zero-shot
clone use). Then uploads the set to the Spark.

    python step4_capture_audio.py --i-am-the-speaker                 # guided session, N takes from studio.yaml
    python step4_capture_audio.py --i-am-the-speaker --import take1.wav take2.wav   # use existing recordings
    python step4_capture_audio.py --i-am-the-speaker --takes 20 --sentences my_sentences.txt
    python step4_capture_audio.py --upload-only

In [ ]:
ARGS = '--i-am-the-speaker'
run('step4_capture_audio.py', ARGS)

## Step 5 - Train (fine-tune) the voice model on your recordings.

Runs on the Spark through the connection. Three passes:
  1. transcripts: any utterance without a transcript (imported recordings) is transcribed with Whisper
  2. prepare:     writes data/train.jsonl ({"audio","text","ref_audio"}) and runs Qwen3-TTS finetuning/prepare_data.py,
                  which encodes the audio with the 12 Hz tokenizer
  3. train:       runs finetuning/sft_12hz.py (single-speaker SFT) -> checkpoints/<run>/checkpoint-epoch-N

The Qwen3-TTS scripts take their arguments by name; the names below follow the project's fine-tuning guide.
If the project changes them, `python finetuning/sft_12hz.py --help` on the Spark shows the current ones
and training.extra_args in studio.yaml passes anything through.

    python step5_train_voice.py --i-am-the-speaker                      # full run, epochs from studio.yaml
    python step5_train_voice.py --i-am-the-speaker --run run2 --epochs 5 --init checkpoints/run1/checkpoint-epoch-3
    python step5_train_voice.py --prepare-only

In [ ]:
ARGS = '--i-am-the-speaker --epochs 3'
run('step5_train_voice.py', ARGS)

## Step 6 - Play back a text file in the trained voice and assess it.

Sends each paragraph of a text file to the model server, saves the audio under out/<name>/, plays each
clip (afplay on macOS, ffplay elsewhere), and asks you to score it 1 to 5 on three things: does it sound
like you, is it natural, and did it say every word right. Words you flag go to data/keywords/wordlist.txt,
which step 7 uses. Scores accumulate in out/assessments.jsonl so runs can be compared.

    python step6_playback_text.py sample.txt
    python step6_playback_text.py sample.txt --no-play          # synthesize only
    python step6_playback_text.py --report                      # mean scores per model/run

In [ ]:
ARGS = 'sample.txt'
run('step6_playback_text.py', ARGS)

## Step 7 - Fine-tune on keywords.

The words the model gets wrong (flagged in step 6, or given here) become targeted training: the script
writes short sentences around each keyword, you record them, they go to data/keywords/, and training
resumes from the last checkpoint with the keyword set included. It also keeps data/keywords/lexicon.json,
a written-form -> spoken-form map, for the words that are better fixed in the text than in the model
(acronyms read as letters, product names respelled); step 6 applies it before speaking.

    python step7_keyword_tune.py --i-am-the-speaker                       # keywords from step 6's wordlist
    python step7_keyword_tune.py --i-am-the-speaker --keywords MBSE SysML Vriesenga --takes-per-keyword 3
    python step7_keyword_tune.py --lexicon MBSE "M B S E" SysML "sis mel"   # text-side fix, no recording
    python step7_keyword_tune.py --i-am-the-speaker --train-only           # recordings exist; just train

In [ ]:
ARGS = '--i-am-the-speaker --keywords MBSE SysML'
run('step7_keyword_tune.py', ARGS)

## Working inside a step

The scripts are importable. To speak one sentence through the loaded model, or to check the connection without the whole step, call the pieces directly.

In [ ]:
from studio.common import Spark, cfg, state
import requests

sp = Spark()
print(sp.run('nvidia-smi --query-gpu=name,memory.used --format=csv,noheader', venv=False, stream=False))
url = sp.url()
print(requests.get(url + '/health', timeout=5).json())
r = requests.post(url + '/speak', json={'text': 'This is a test of the studio from the notebook.'}, timeout=300)
open('out/notebook_test.wav', 'wb').write(r.content); print('out/notebook_test.wav', r.headers.get('X-RTF'), 'x real time')
from IPython.display import Audio; Audio('out/notebook_test.wav')

## Assessment report

Mean scores per model from step 6, so a fine-tuned checkpoint can be compared with the zero-shot clone on the same text.

In [ ]:
run('step6_playback_text.py', '--report')